<a href="https://colab.research.google.com/github/AdiDev1411/Machine-learning/blob/main/Optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install optuna

In [6]:
import opendatasets as op

In [21]:
import pandas as pd
import numpy as np
import optuna

In [10]:
df = pd.read_csv('diabetes.csv')

In [11]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [12]:
cols_with_missing_values = ['SkinThickness','Insulin','BMI','Glucose','BloodPressure']
df[cols_with_missing_values] = df[cols_with_missing_values].replace(0,np.nan)

df.fillna(df.mean(),inplace=True)
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [15]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]


In [16]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split( X , y  , test_size=0.3 , random_state=42)

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [18]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit(X_test)

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [22]:
def objective(trial):
  n_estimators = trial.suggest_int('n_estimators' , 50 ,200)
  max_depth = trial.suggest_int('max_depth' , 3, 20)

  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )

  score = cross_val_score( model , X_train , y_train , cv=3 , scoring='accuracy').mean()

  return score

In [23]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2025-07-30 08:34:31,160] A new study created in memory with name: no-name-cb836598-23af-4b4d-9750-70f164d6df58
[I 2025-07-30 08:34:31,791] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 106, 'max_depth': 10}. Best is trial 0 with value: 0.7616387337057727.
[I 2025-07-30 08:34:32,288] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 82, 'max_depth': 9}. Best is trial 0 with value: 0.7616387337057727.
[I 2025-07-30 08:34:33,228] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 162, 'max_depth': 8}. Best is trial 2 with value: 0.7653631284916201.
[I 2025-07-30 08:34:33,793] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 97, 'max_depth': 11}. Best is trial 2 with value: 0.7653631284916201.
[I 2025-07-30 08:34:34,696] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 155, 'max_depth': 10}. Best is trial 4 with value: 0.76908752

In [24]:
print(f'best trail accuracy : {study.best_trial.value}')
print(f'best trail accuracy : {study.best_trial.params}')

best trail accuracy : 0.7821229050279329
best trail accuracy : {'n_estimators': 125, 'max_depth': 18}
